# Evaluate event recovery

In [1]:
# Import the analysis tools and record the package version.
import nahiku
from nahiku import Nahiku
from nahiku.search import evaluate_events
print(f"Nāhiku {nahiku.__version__}")

/tmp/nahiku-02/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Nāhiku 0.2.0


In [2]:
# Generate reproducible observations and label two injected events.
lc = Nahiku.from_synthetic_noise(duration=10, n_points=200, period=2,
                                 amplitude=0.3, white_noise=0.02, red_noise=0, seed=48)
lc.inject_anomaly(center=4.8, width=0.12, depth=0.8, shape="gaussian")
lc.inject_anomaly(center=7.4, width=0.07, depth=0.45, shape="exocomet")

# Standardize the data and fit the shared GP.
lc.standardize()
lc.fit_gp(period=2, length_scale=4, iterations=120)

# Run both searches independently with their specified selection rules.
greedy = lc.greedy_search(sigma=4, grow_sigma=2, seed_statistic="loo", growth="threshold",
                         direction="negative", gap_factor=5, max_events=20, backend="numpy")
exhaustive = lc.exhaustive_search(min_points=3, max_points=16, step=3,
                                size_step=3, threshold=1e-8)

# Match detections to injected truth, allowing one sample of boundary tolerance.
for result in (greedy, exhaustive):
    print(result.method, lc.evaluate(result, buffer=1))

greedy {'matched_events': 2, 'detected_events': 2, 'true_events': 2, 'false_positives': 0, 'missed_events': 0, 'precision': 1.0, 'recall': 1.0}
exhaustive {'matched_events': 2, 'detected_events': 5, 'true_events': 2, 'false_positives': 3, 'missed_events': 0, 'precision': 0.4, 'recall': 1.0}


In [3]:
# One broad detection cannot receive credit for two separate true events.
report = evaluate_events(detected=[(2, 10)], truth=[(3, 5), (7, 9)])
print("Broad detection:", report)
assert report["matched_events"] == 1
assert report["missed_events"] == 1

# Half-open intervals that only touch do not overlap without a buffer.
for buffer in (0, 1):
    report = evaluate_events(detected=[(2, 4)], truth=[(4, 6)], buffer=buffer)
    print(f"Boundary tolerance {buffer}:", report)

Broad detection: {'matched_events': 1, 'detected_events': 1, 'true_events': 2, 'false_positives': 0, 'missed_events': 1, 'precision': 1.0, 'recall': 0.5}
Boundary tolerance 0: {'matched_events': 0, 'detected_events': 1, 'true_events': 1, 'false_positives': 1, 'missed_events': 1, 'precision': 0.0, 'recall': 0.0}
Boundary tolerance 1: {'matched_events': 1, 'detected_events': 1, 'true_events': 1, 'false_positives': 0, 'missed_events': 0, 'precision': 1.0, 'recall': 1.0}


## Interpret the result

With a one-sample overlap tolerance, greedy search matches both injected events without extra detections (precision and recall 1.0). Exhaustive search also matches both events but reports three extra detections (precision 0.4, recall 1.0). These recovery measurements apply to this simulated light curve and configuration. Survey performance requires separate calibration.

Evaluation uses maximum one-to-one overlap matching. Precision is matched events divided by detected events; recall is matched events divided by true events. A single broad detection spanning two truth events earns only one match, as the explicit interval example demonstrates. If a denominator is zero, the corresponding metric is reported as zero.

Intervals use half-open sample bounds `[start, stop)`. Adjacent intervals that only touch do not overlap. `buffer=1` expands each truth interval by one sample on either side for matching; it does not change the detections. The tolerance is measured in samples, not days, so its time span depends on sampling.

`lc.evaluate` uses accumulated injection labels unless a boolean `truth` mask is supplied. Event-level overlap does not measure how accurately a detection recovers an event's boundaries, depth, or shape. Real observations require justified truth labels; a known published dip does not establish a complete truth catalogue.

Consult the API reference for parameter and return-value details.